# Titanic dimensionality reduction: Live coding scaffold

**Question:** What profiles appear across passenger characteristics, and how do they relate to survival?

Preparation and plotting helpers are supplied. Fill in the method calls live. Saved runs are loaded as a fallback, and this scaffold has no saved outputs.

The main live path is preparation, one fit_transform call per method, recoloring, and a return to original variables. The SVD and parameter sections are optional follow-up.

**Saved-run provenance:** PCA and t-SNE use scikit-learn, PHATE uses its Python implementation, and UMAP uses the official UMAP-js implementation. The Python UMAP recomputation branch uses umap-learn and was not executable in the authoring environment. The two implementations can produce different layouts. See [run details](../run_details.qmd).


## Setup

Run before class. The four saved runs are already computed for the bundled data. Install the recompute extra before using UMAP or PHATE fitting code.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

ROOT = Path.cwd()
if not (ROOT / "support/eda_helpers.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "support"))
from eda_helpers import prepare_titanic, prepare_tornadoes, load_saved, load_variant, plot_embedding

# False uses supplied runs and requires only the base environment.
# True recomputes nonlinear methods. First run: uv sync --extra recompute
RECOMPUTE = False


## Prepared features and plotting metadata

The supplied preparation function imputes numerical fields by the median, standardizes them, and imputes/one-hot encodes categorical fields. It retains all 891 passengers. Open `support/eda_helpers.py` to inspect the full pipeline.

| Embedding inputs | Preparation |
|---|---|
| age, fare, sibsp, parch | Median imputation, then z-scores |
| pclass, sex, embarked | Most-frequent imputation, then one-hot encoding |
| survived | Excluded from inputs, retained for color |
| Missingness indicators | Excluded from inputs, retained for color |

The numerical block and each categorical block contribute differently to Euclidean distances. This is one explicit definition of passenger similarity, not an automatically optimal choice.


In [ ]:
df, X, feature_names, preprocessor = prepare_titanic(ROOT)
embeddings = {} if RECOMPUTE else load_saved(ROOT, "titanic", X, df["row_id"])
print("Prepared matrix:", X.shape)
print("Feature columns:", feature_names)
assert "survived" not in feature_names
df[["row_id", "age", "fare", "pclass", "sex", "survived"]].head()


## PCA: a linear baseline

Before running: what kind of information does this method prioritize?


In [ ]:
# TODO: Create PCA with two components, then fit_transform(X).
# Replace embeddings["PCA"] with the resulting coordinates.
# Optional: inspect the explained_variance_ratio_.

# The saved run remains available if you skip a computation.
plot_embedding(embeddings["PCA"], df, title="PCA: passenger profiles")
plt.tight_layout()
plt.show()


**Discuss:** What does this view suggest? What would you color or inspect next?


## t-SNE: neighborhood similarities

Before running: what kind of information does this method prioritize?


In [ ]:
# TODO: If recomputing, create TSNE and fit_transform(X).
# Suggested settings: perplexity=30, init="pca", learning_rate="auto",
# max_iter=1000, random_state=42.
# Replace embeddings["t-SNE"] with the coordinates.

plot_embedding(embeddings["t-SNE"], df, "sex", "t-SNE colored by recorded sex")
plt.tight_layout()
plt.show()


**Discuss:** What does this view suggest? What would you color or inspect next?


## UMAP: a neighborhood graph

Before running: what kind of information does this method prioritize?


In [ ]:
# TODO: If recomputing, import umap and create umap.UMAP(...).
# Suggested settings: n_neighbors=15, min_dist=0.1, random_state=42.
# Replace embeddings["UMAP"] with fit_transform(X).

plot_embedding(embeddings["UMAP"], df, "pclass", "UMAP colored by passenger class")
plt.tight_layout()
plt.show()


**Discuss:** What does this view suggest? What would you color or inspect next?


## PHATE: diffusion and broader structure

Before running: what kind of information does this method prioritize?


In [ ]:
# TODO: If recomputing, import phate and create phate.PHATE(...).
# Suggested settings: knn=30, t="auto", mds_solver="smacof", random_state=42.
# Replace embeddings["PHATE"] with fit_transform(X).

plot_embedding(embeddings["PHATE"], df, "survived", "PHATE colored by survival")
plt.tight_layout()
plt.show()


**Discuss:** What does this view suggest? What would you color or inspect next?


## Recolor the same coordinates

Choose a method and a variable, then rerun this cell. Recoloring does not refit the embedding. Compare an input feature, a missingness indicator, and survival, which was withheld.

Options: sex, pclass, age, fare, age_is_missing, embarked_is_missing, survived.


In [ ]:
METHOD = "UMAP"
COLOR_BY = "survived"
plot_embedding(embeddings[METHOD], df, COLOR_BY, f"{METHOD}: {COLOR_BY}")
plt.tight_layout()
plt.show()


## A pattern becomes another question

Does apparent survival separation simply reflect passenger class and sex? Return to the original variables and compare proportions with explicit denominators.


In [ ]:
# TODO: Group passengers by class and sex.
# Show group sizes, survivor counts, and survival proportions.
# What does this original-variable view add to the embedding?


## Optional: inspect a region and compare it with the remaining passengers

This example takes the rightmost 20% of one layout as a convenient inspection region. It does not define a validated cluster. Axis direction is arbitrary and the region changes if the embedding rotates or changes settings.


In [ ]:
coords = embeddings["UMAP"]
in_region = coords[:, 0] > np.quantile(coords[:, 0], 0.8)
regions = df.assign(region=np.where(in_region, "Selected region", "Remaining passengers"))
display(regions.groupby("region").agg(
    n=("row_id", "size"), survival=("survived", "mean"),
    average_fare=("fare", "mean"), missing_age=("age_is_missing", "mean")
).round(3))
display(pd.crosstab(regions["region"], regions["pclass"], normalize="index").round(3))


## Optional: connect SVD to PCA

For centered X, X = U diag(s) Vt. The first two right singular vectors define directions. U[:, :2] * s[:2] gives point coordinates. A component's sign can flip without changing the represented geometry.


In [ ]:
X_centered = X - X.mean(axis=0)
U, s, Vt = np.linalg.svd(X_centered, full_matrices=False)
svd_scores = U[:, :2] * s[:2]
variance = s**2 / (len(X) - 1)
print("Variance fraction from SVD:", variance[:2].sum() / variance.sum())
display(pd.DataFrame(Vt[:2].T, index=feature_names, columns=["PC1 weight", "PC2 weight"]).round(3))
print("Scores shape:", svd_scores.shape)


## Optional: a controlled parameter comparison

These supplied runs hold the dataset and preprocessing fixed. The example below compares t-SNE perplexity 5 with 30. Use the same color variable and avoid reading between-island distances as original distances.


In [ ]:
alternative = load_variant(ROOT, "tsne_perplexity_5", df["row_id"], X)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_embedding(alternative, df, "survived", "Perplexity 5", axes[0])
plot_embedding(load_variant(ROOT, "tsne", df["row_id"], X), df, "survived", "Perplexity 30", axes[1])
plt.tight_layout()
plt.show()


## Optional: UMAP and PHATE sensitivity

Other supplied variants: umap_neighbors_5, umap_neighbors_50, umap_min_dist_08, umap_seed_7, phate_t_2, phate_t_30. The source settings and implementation details appear in the run manifest.

PHATE's t changes diffusion smoothing. It is not a time variable measured on the passengers. Changing settings is a sensitivity check rather than a search for the prettiest plot.

**Next question:** Which interpretation survives a reasonable alternative view and an original-variable check?
